In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

model = xgb.Booster()
model.load_model('loan_repayment_model_day1.json') 
test = pd.read_csv('data/test.csv')
test_ids = test['id']

def prepare_submission_data(df):
    education_ranking = {"PhD": 5, "Master's": 4, "Bachelor's": 3, "High School": 2, "Other": 1}
    df['education_rank'] = df['education_level'].map(education_ranking)
    df['subgrade_num'] = df['grade_subgrade'].str[1:].astype(int)
    grade_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6}
    df['grade_rank'] = df['grade_subgrade'].str[0].map(grade_map)
    df = pd.get_dummies(df, columns=['gender', 'marital_status', 'employment_status', 'loan_purpose'], drop_first=True)
    cols_to_drop = ['id', 'education_level', 'grade_subgrade']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    return df

X_test = prepare_submission_data(test)
expected_features = model.feature_names
X_test = X_test.reindex(columns=expected_features, fill_value=0)

dtest = xgb.DMatrix(X_test)
test_preds = model.predict(dtest)

submission = pd.DataFrame({'id': test_ids, 'loan_paid_back': test_preds})
submission.to_csv('submission_day1.csv', index=False)